In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm
import os
import time
import argparse
import sys

device = 'cuda' if torch.cuda.is_available() else 'cpu'
classes = ('airplane', 'automobile', 'bird', 'cat', 'deer',
               'dog', 'frog', 'horse', 'ship', 'truck')

# ImageNet 통계
MEAN = (0.5, 0.5, 0.5)
STD = (0.2, 0.2, 0.2)
IMAGE_SIZE = 224
# Vit 244 x 244
# 학습용 증강 파이프라인
train_transform = transforms.Compose([
    transforms.RandomResizedCrop(IMAGE_SIZE, IMAGE_SIZE), #    transforms.RandomResizedCrop(IMAGE_SIZE, scale=(0.8, 1.0)), 
    transforms.RandomHorizontalFlip(p=0.5),                      # 좌우 반전
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),  # 색상 변형
    transforms.RandomRotation(degrees=15),                       # 회전
    transforms.ToTensor(),
    transforms.Normalize(mean=MEAN, std=STD),
])
    
# 평가용 파이프라인 (증강 없음)
val_transform = transforms.Compose([
    transforms.Resize(256),
    # transforms.CenterCrop(IMAGE_SIZE),
    transforms.ToTensor(),
    transforms.Normalize(mean=MEAN, std=STD),
])

train_dataset = datasets.CIFAR10(root='./data', train=True, download=True, transform=train_transform)
test_dataset = datasets.CIFAR10(root='./data', train=False, download=True, transform=val_transform)

# 학습 검증 분리 (10% 검증)
from sklearn.model_selection import train_test_split
train_dataset, val_dataset = train_test_split(train_dataset, test_size=0.1, random_state=42)

# 데이터 로더
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

def prepare_vit_model(num_classes:int=10, device:str='cpu', strategy:str='full') :
    '''[strategy]
        linear_probe: 분류 head만 사용 (transformer encoder 및 patch embedding 부분 동결) ---> classifier 사용
        partial: 일부 레이어 활성, classifier 사용
        full : 동결없음, 전체 파라미터 학습
    '''
    # Vit 모델 로드
    from transformers import ViTForImageClassification, ViTConfig
    model_name = 'google/vit-base-patch16-224'
    model = ViTForImageClassification.from_pretrained(
        model_name,
        num_labels = 10, 
        ignore_mismatched_sizes=True
    )

    # 파인튜닝 전략
    if strategy == 'linear_probe':
        print(f"\n파인튜닝전략 : Linear Probling")
        for param in model.vit.parameters():
            param.requires_grad = False
        print('ViT Encoder 동결')
        print('Classifier 학습')
    elif strategy == 'partial':
        for param in model.vit.parameters():
            param.requires_grad = False
        for param in model.vit.encoder.layer[-2:].parameters():
            param.requires_grad = True
        print(f'ViT Encoder 레이어 마지막 2개를 제외 : 동결')
        print(f'ViT Encoder 레이어 마지막 2개 : 학습')
        print('classifier 학습')
    elif strategy == 'full':
        trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
        total_params = sum(p.numel() for p in model.parameters())
        print(f'전체 파라미터는 : {total_params}')
        print(f"학습 파라미터는 : {trainable_params}")
        print(f"학습 비율 : {trainable_params/total_params}")
    model = model.to(device)
    return model

100%|██████████| 170M/170M [00:35<00:00, 4.80MB/s] 
